# Odds Ratio Calculation for Functional Groups

In [1]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

In [2]:
# Constants (mostly paths to things)

functional_carriers_path = 'carrier_tables/functional-carriers_2026-10-08.parquet'
af_map_path = 'carrier_tables/af_maps/functional_2026-10-08.parquet'
variant_index_path = 'carrier_tables/var_maps/functional_2026-10-08.parquet'
demographics_path = '../cohorts/demo.tsv.gz'
ancestry_pca_path = '../cohorts/ancestry_pca.tsv.gz'
or_estimates_path = 'results/functional-OR-estimates_2026-10-07.parquet'

# Index column labels
index_cols=['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

In [3]:
carriers_df = pd.read_parquet(functional_carriers_path)
carriers_df

OR calculation

In [4]:
demographics_df = pd.read_table(demographics_path, index_col='person_id')
demographics_df

In [5]:
pca_df = pd.read_table(ancestry_pca_path, index_col='research_id')
pca_df

In [6]:
from functools import cache

@cache
def assemble_cohort(gene):
    result = pd.read_parquet(f'gene_cohorts/{gene} cohort.parquet').join(demographics_df).join(pca_df)
    # Trichotomize sex
    result.sex_at_birth = result.sex_at_birth.where(
        result.sex_at_birth.isin(['Male', 'Female']),
        'Other'
    )
    
    return result


In [7]:
from cvfgaou import odds

def compute_OR_and_add_stats(exposure_df, gene):
    exposure_series = pd.to_numeric(exposure_df.index.to_frame().s, downcast='integer')
    cohort_df = assemble_cohort(gene)
    return odds.estimate_logOR(
        pd.to_numeric(exposure_df.index.to_frame().s, downcast='integer'),
        assemble_cohort(gene)
    ) | odds.collect_variant_stats(
        pd.to_numeric(exposure_df.index.to_frame().s, downcast='integer'),
        assemble_cohort(gene),
        exposure_df.variants
    )

or_estimates_dict = {
    (dataset, gene, grouping, consequence, group):
        compute_OR_and_add_stats(exposure_df, gene)    
    for (dataset, gene, grouping, consequence, group), exposure_df
    in tqdm(carriers_df.groupby(index_cols))
}

In [8]:
or_estimates_df = pd.DataFrame.from_dict(or_estimates_dict, orient='index')
or_estimates_df.index.set_names(index_cols, inplace=True)
or_estimates_df

In [9]:
or_estimates_df.to_parquet(or_estimates_path)